# Preprocess Collected Data
* Preprocess the collected data
* Label the collected data

In [1]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from transformers import pipeline
import torch
from collections import Counter
from deep_translator import GoogleTranslator

c:\Users\yuyla\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Data Ingestion and Inspection

In [3]:
df = pd.read_csv("../data/latest_malaysia_food_comments.csv")
# df = pd.read_csv("../data/latest_malaysia_food_comments.csv", usecols=['comment_id', 'comment'])
df.head()

,comment_id,video_id,video_title,video_url,channel_title,author,comment,likes,reply_count,comment_length,published_at,updated_at
0,UgzaWfPbrkBJ2IAnmnd4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@spicytrinirice,Mark how are you so skinny?,0,0,27,2026-06-13T04:23:22Z,2026-06-13T04:23:22Z
1,Ugw_Y1XANSnQcxzrG1x4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@WaqasAli-dq9wg,Kuala Lumpur rocks! I think I&#39;m gonna be r...,0,0,105,2026-06-02T21:32:29Z,2026-06-02T21:32:29Z
2,UgyvzylHLtmBKoWD0rJ4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@ruohuailiu,It makes my mouth water.,0,0,24,2026-05-27T03:41:56Z,2026-05-27T03:41:56Z
3,Ugy8SZOGQ-vroxWT0QV4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@AhhiongChia,Why not batu Kawa,0,0,17,2026-05-26T08:34:31Z,2026-05-26T08:34:31Z
4,UgxrG1FxMEOFzZp1rPJ4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@wolflamb632,I loooove lapcheong.,0,0,20,2026-05-26T07:45:17Z,2026-05-26T07:45:17Z


In [4]:
# Data Inspection
print(df.shape)
print()
print(df.dtypes)
print()
print(df.isnull().sum())

(37937, 12)

comment_id          str
video_id            str
video_title         str
video_url           str
channel_title       str
author              str
comment             str
likes             int64
reply_count       int64
comment_length    int64
published_at        str
updated_at          str
dtype: object

comment_id        0
video_id          0
video_title       0
video_url         0
channel_title     0
author            0
comment           0
likes             0
reply_count       0
comment_length    0
published_at      0
updated_at        0
dtype: int64


In [5]:
df = df.drop_duplicates()
print(df.shape)

(37932, 12)


### Data Preprocessing

In [ ]:
# Data Preprocessing
def preprocess_comment(text):
    if not isinstance(text, str):
        return ""
    
    # Strip HTML tags
    text = BeautifulSoup(text, 'html.parser').get_text()
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Replace mentions and hashtags
    text = re.sub(r'@\w+', '@user', text)
    text = re.sub(r'#\w+', '', text)
    
    # Remove HTML entities
    text = re.sub(r'&\w+;', '', text)
    
    # Normalise repeated characters
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['comment_clean'] = df['comment'].apply(preprocess_comment)

C:\Users\yuyla\AppData\Local\Temp\ipykernel_25788\3809786508.py:7: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  text = BeautifulSoup(text, 'html.parser').get_text()


In [7]:
# Filter Noise
def is_garbage(text):
    if not isinstance(text, str):
        return True
    # Mostly numbers/symbols (no real word characters)
    if re.sub(r'[\d\s\W]', '', text) == '':
        return True
    # Too few actual characters after stripping non-alphanumeric
    if len(re.sub(r'[^\w]', '', text)) < 3:
        return True
    # Repeated single word (e.g. "haha haha haha")
    words = text.split()
    if len(words) >= 3 and len(set(words)) == 1:
        return True
    # Spam-like: excessive punctuation/caps ratio
    if len(text) > 0 and sum(1 for c in text if c in '!?') / len(text) > 0.3:
        return True
    return False

# Drop nulls and empty strings
df = df[df['comment_clean'].str.len() > 0].copy()

# Drop duplicates
df = df.drop_duplicates(subset='comment_clean').reset_index(drop=True)

# Drop garbage
df = df[~df['comment_clean'].apply(is_garbage)].reset_index(drop=True)

# Drop very short comments
df = df[df['comment_clean'].str.split().str.len() >= 3].reset_index(drop=True)

# Drop very long comments (outliers that may break tokenizer max_length)
df = df[df['comment_clean'].str.split().str.len() <= 200].reset_index(drop=True)

print(f"Clean dataset size: {len(df)}")

Clean dataset size: 32368


In [8]:
df[['comment', 'comment_clean']].sample(10)

,comment,comment_clean
21122,Mark wiens never disappointed me,mark wiens never disappointed me
20600,"If you&#39;re still there, Mollies Desserts at...","if you're still there, mollies desserts at the..."
16334,He must try indonesian food !,he must try indonesian food !
24148,that was nowhere near the spicyest stuff you w...,that was nowhere near the spicyest stuff you w...
19551,I’m form Malaysia 🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾...,i’m form malaysia 🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾🇲🇾...
18772,Mark must have done his homework beforehand......,mark must have done his homework beforehand..a...
5279,try spending rm100 instead of $100. 😂 😂 😂,try spending rm100 instead of $100. 😂 😂 😂
578,"more malaysia/se asia, mark","more malaysia/se asia, mark"
1462,My mum LOVES durian. I tried it too. But for t...,my mum loves durian. i tried it too. but for t...
17079,Enakan makanan sate indo suarez coba aja,enakan makanan sate indo suarez coba aja


In [17]:
from langdetect import detect

def is_english(text):
    try:
        return detect(str(text)) == 'en'
    except:
        return False  # if detection fails, assume needs translation

# Identify rows that need translation
needs_translation = ~df["comment_clean"].apply(is_english)
print(f"Rows needing translation: {needs_translation.sum()} / {len(df)}")

Rows needing translation: 9226 / 32368


In [19]:
import time

def translate_with_retry(text, retries=3, delay=2):
    for attempt in range(retries):
        try:
            return GoogleTranslator(source='auto', target='en').translate(str(text))
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(delay)
            else:
                print(f"Failed: {str(text)[:30]}... | Error: {e}")
                return text  # keep original if all retries fail

# Translate data into english
df["comment_en"] = df["comment_clean"].copy()
df.loc[needs_translation, "comment_en"] = df.loc[needs_translation, "comment_clean"].apply(
    translate_with_retry
)

In [20]:
df[['comment_clean', 'comment_en']].sample(10)

,comment_clean,comment_en
27104,malaysia has really good food and tradional,malaysia has really good food and tradional
28527,im from malaysia 🇲🇾🇲🇾,im from malaysia 🇲🇾🇲🇾
20170,chicken satay for sure and dessert,chicken satay for sure and dessert
7740,is the dude traveling with you single? proud t...,is the dude traveling with you single? proud t...
18418,"23:46 ""god's given sauce"" whoa 😮 nice sentence...","23:46 ""god's given sauce"" whoa 😮 nice sentence..."
12222,nasi lemak kok ngga ada harganya ya?,How come there's no price for nasi lemak?
6404,itu pulak saya makan😅😅,that's what I eat😅😅
23780,can't lie those noodles look soo good,can't lie those noodles look soo good
13637,ya ampun smpe nelen ludah liat bang tanboy kun...,"Oh my gosh, I'm salivating when I see Tanboy K..."
24274,actually it is fish pulusu in ap india,actually it is fish pulusu in ap india


In [21]:
df.sample(10)

,comment_id,video_id,video_title,video_url,channel_title,author,comment,likes,reply_count,comment_length,published_at,updated_at,comment_clean,comment_en
18843,Ugw4Pfi3Fb4fdT-9BER4AaABAg,_lZjm3mMl3g,Malaysia STREET FOOD Heaven!! 🇲🇾 27 Meals Best...,https://www.youtube.com/watch?v=_lZjm3mMl3g,Mark Wiens,@ubbgn,embarrassing!<br>massive bites eating like a 🐷🐖!,0,0,49,2025-09-07T17:50:35Z,2025-09-07T19:19:36Z,embarrassing!massive bites eating like a 🐷🐖!,embarrassing!massive bites eating like a 🐷🐖!
1871,UgxXrRP8dsI_K-iC75d4AaABAg,tG8ZaAgCTiE,Trying Viral Malaysian Street Food 🇲🇾,https://www.youtube.com/watch?v=tG8ZaAgCTiE,Safiya Nygaard,@amberwebster83,At an Asian grocery store I went to the had du...,0,0,114,2025-03-16T13:24:17Z,2025-03-16T13:24:17Z,at an asian grocery store i went to the had du...,at an asian grocery store i went to the had du...
10638,UgxGZsfe8kS1dcrpLlJ4AaABAg,6-gyPec-NSA,TRYING MALAYSIAN FOOD FOR THE FIRST TIME!! Nas...,https://www.youtube.com/watch?v=6-gyPec-NSA,Cheat Day Around The World,@unknow-r6k,Welcome to Malaysia 🇲🇾 hope you enjoy ❤️,1,0,40,2022-11-24T08:58:16Z,2022-11-24T08:58:16Z,welcome to malaysia 🇲🇾 hope you enjoy ❤️,welcome to malaysia 🇲🇾 hope you enjoy ❤️
21929,Ugz1X0j9_p4fX0WVjWJ4AaABAg,feDph2WgRfI,The food adventure MY STOMACH WILL NEVER FORGET!,https://www.youtube.com/watch?v=feDph2WgRfI,Blondie in China,@LimLim-g2t,You rediscovered my childhood! I&#39;m living ...,3,0,133,2025-10-14T06:02:18Z,2025-10-14T06:02:18Z,you rediscovered my childhood! i'm living in v...,you rediscovered my childhood! i'm living in v...
21784,UgzqH5QOzW5fL1oSYyl4AaABAg,feDph2WgRfI,The food adventure MY STOMACH WILL NEVER FORGET!,https://www.youtube.com/watch?v=feDph2WgRfI,Blondie in China,@hansiejohnson,I’m not here to do WHAT with spiders??? Hahaha...,0,0,132,2025-10-14T14:33:19Z,2025-10-14T14:33:19Z,i’m not here to do what with spiders?? hahahah...,i’m not here to do what with spiders?? hahahah...
28946,UgxTuqRlJqJLsKjxfzJ4AaABAg,DBs3YyNh8xg,Trying Food From Malaysia 🇲🇾😋,https://www.youtube.com/watch?v=DBs3YyNh8xg,Junpei Zaki,@amayaayris,Try the Malaysia flag the named Kedah and try ...,0,0,59,2023-12-18T08:40:14Z,2023-12-18T08:40:14Z,try the malaysia flag the named kedah and try ...,try the malaysia flag the named kedah and try ...
18504,Ugz15H2BrSFsrokznbV4AaABAg,K7DeqmLxVFM,Foreigner can't stop OVEREATING Malaysian food...,https://www.youtube.com/watch?v=K7DeqmLxVFM,Daily Max,@SSxxx18,"<a href=""https://www.youtube.com/watch?v=K7Deq...",0,0,110,2024-02-24T17:53:32Z,2024-02-24T17:53:32Z,38:45yes. in it's simplest form !,38:45yes. in it's simplest form !
28593,Ugxr55c-q1ToVSopYDZ4AaABAg,DBs3YyNh8xg,Trying Food From Malaysia 🇲🇾😋,https://www.youtube.com/watch?v=DBs3YyNh8xg,Junpei Zaki,@ZS_ANIQ,The Malaysia stay palestine and Malaysia Pales...,0,0,83,2024-02-16T17:33:45Z,2024-02-16T17:33:45Z,the malaysia stay palestine and malaysia pales...,the malaysia stay palestine and malaysia pales...
18824,UgyPt_NRlYGDuW0jPRh4AaABAg,_lZjm3mMl3g,Malaysia STREET FOOD Heaven!! 🇲🇾 27 Meals Best...,https://www.youtube.com/watch?v=_lZjm3mMl3g,Mark Wiens,@Mzxcy,"<a href=""https://www.youtube.com/watch?v=_lZjm...",0,1,125,2025-10-31T04:37:28Z,2025-10-31T04:37:28Z,12:48 are u okay. yeah bro is definitely lying...,12:48 are u okay. yeah bro is definitely lying...
17228,Ugy5twrGjjXu47JBek14AaABAg,6I4g9qI72o0,Malaysian Cuisine,https://www.youtube.com/watch?v=6I4g9qI72o0,Malaysia Truly Asia,@rekardo8067,INDOGS=INDO GAMERS :),43,50,21,2018-04-16T15:31:09Z,2018-04-16T15:31:09Z,indogs=indo gamers :),indogs=indo gamers :)


### Label Data

In [22]:
device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU' if device == 0 else 'CPU'}")

Using: CPU


In [23]:
# Model 1: Twitter-RoBERTa
roberta_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device,
    truncation=True,
    max_length=512
)

# Model 2: XLM-RoBERTa
xlmr_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=device,
    truncation=True,
    max_length=512
)

# Model 3: Multilingual 
multi_pipe = pipeline(
    "sentiment-analysis",
    model="tabularisai/multilingual-sentiment-analysis",
    tokenizer="tabularisai/multilingual-sentiment-analysis",
    device=device,
    truncation=True,
    max_length=512
)

# # Model 4: Malaya (got issues)
# malaya_model = malaya.sentiment.huggingface(
#     model='mesolitica/sentiment-analysis-nanot5-small-malaysian-cased'
# )

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 22914.71it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1118.56it/s]


In [24]:
from tqdm import tqdm

BATCH_SIZE = 32

def run_pipeline_batched(pipe, texts):
    results = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch = texts[i:i+BATCH_SIZE]
        batch_results = pipe(batch, truncation=True, max_length=512)
        results.extend(batch_results)
    return results

# def run_malaya_batched(model, texts):
#     results = []
#     for i in tqdm(range(0, len(texts), BATCH_SIZE)):
#         batch = texts[i:i+BATCH_SIZE]
#         probs = model.predict_proba(batch)
#         for prob in probs:
#             label = max(prob, key=prob.get)
#             score = prob[label]
#             results.append({'label': label, 'score': score})
#     return results

In [25]:
texts = df['comment_clean'].tolist()

print("Running Twitter-RoBERTa...")
roberta_results = run_pipeline_batched(roberta_pipe, texts)

print("Running XLM-RoBERTa...")
xlmr_results = run_pipeline_batched(xlmr_pipe, texts)

print("Running Multilingual...")
multi_results = run_pipeline_batched(multi_pipe, texts)

# print("Running Malaya...")
# malaya_results = run_malaya_batched(malaya_model, texts)

Running Twitter-RoBERTa...


100%|██████████| 1012/1012 [24:31<00:00,  1.45s/it]


Running XLM-RoBERTa...


100%|██████████| 1012/1012 [23:39<00:00,  1.40s/it]


Running Multilingual...


100%|██████████| 1012/1012 [12:35<00:00,  1.34it/s]


In [26]:
def normalise_label(label: str) -> str:
    label = label.lower().strip()
    
    if label == 'very positive':
        return 'positive'
    if label == 'very negative':
        return 'negative'
    
    return label

df['roberta_label'] = [normalise_label(r['label']) for r in roberta_results]
df['roberta_score'] = [r['score'] for r in roberta_results]

df['xlmr_label'] = [normalise_label(r['label']) for r in xlmr_results]
df['xlmr_score'] = [r['score'] for r in xlmr_results]

df['multi_label'] = [normalise_label(r['label']) for r in multi_results]
df['multi_score'] = [r['score'] for r in multi_results]

# df['malaya_label'] = [normalise_label(r['label']) for r in malaya_results]
# df['malaya_score'] = [r['score'] for r in malaya_results]

In [27]:
# Voting
def vote(row):
    labels = [row['roberta_label'], row['xlmr_label'], row['multi_label']]
    scores = {
        row['roberta_label']: row['roberta_score'],
        row['xlmr_label']:    row['xlmr_score'],
        row['multi_label']:  row['multi_score'],
    }
    
    count = Counter(labels)
    most_common_label, most_common_count = count.most_common(1)[0]
    
    if most_common_count == 3:
        agreement = 'full'
        confidence = np.mean([row['roberta_score'], row['xlmr_score'], row['multi_score']])
    elif most_common_count == 2:
        agreement = 'majority'
        # Average confidence of the 2 agreeing models only
        agreeing_scores = [
            score for label, score in zip(
                [row['roberta_label'], row['xlmr_label'], row['multi_label']],
                [row['roberta_score'], row['xlmr_score'], row['multi_score']]
            ) if label == most_common_label
        ]
        confidence = np.mean(agreeing_scores)
    else:
        agreement = 'none'
        most_common_label = 'review'
        confidence = 0.0
    
    return pd.Series({
        'final_label': most_common_label,
        'confidence':  round(confidence, 4),
        'agreement':   agreement
    })

df[['final_label', 'confidence', 'agreement']] = df.apply(vote, axis=1)

In [28]:
print(df['agreement'].value_counts())
print(df['final_label'].value_counts())

# # Flag low confidence majority cases for manual review (gave up manual review)
# manual_review = df[
#     (df['agreement'] == 'none') 
#     | ((df['agreement'] == 'majority') & (df['confidence'] < 0.6))
# ]
# print(f"\nSamples needing manual review: {len(manual_review)}")
# manual_review[['comment_clean', 'roberta_label', 'xlmr_label', 'multi_label', 'final_label', 'confidence']].sample(10)

agreement
full        15659
majority    15076
none         1633
Name: count, dtype: int64
final_label
positive    13537
neutral     10578
negative     6620
review       1633
Name: count, dtype: int64


In [ ]:
high_conf = df[
    (df['agreement'] == 'full') |
    ((df['agreement'] == 'majority') & (df['confidence'] >= 0.6))
].copy()

high_conf = high_conf.rename(columns={'final_label': 'label'})

# Standardise labels
label_map_str = {'positive': 2, 'neutral': 1, 'negative': 0}
high_conf['label'] = high_conf['label'].map(label_map_str)

high_conf.head()

,comment_id,video_id,video_title,video_url,channel_title,author,comment,likes,reply_count,comment_length,...,comment_en,roberta_label,roberta_score,xlmr_label,xlmr_score,multi_label,multi_score,label,confidence,agreement
0,UgzaWfPbrkBJ2IAnmnd4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@spicytrinirice,Mark how are you so skinny?,0,0,27,...,mark how are you so skinny?,neutral,0.628314,negative,0.879051,neutral,0.673817,1,0.6511,majority
1,Ugw_Y1XANSnQcxzrG1x4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@WaqasAli-dq9wg,Kuala Lumpur rocks! I think I&#39;m gonna be r...,0,0,105,...,kuala lumpur rocks! i think i'm gonna be reall...,positive,0.989310,positive,0.891053,positive,0.585007,2,0.8218,full
3,Ugy8SZOGQ-vroxWT0QV4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@AhhiongChia,Why not batu Kawa,0,0,17,...,why not batu kawa,neutral,0.654108,neutral,0.484825,neutral,0.512170,1,0.5504,full
4,UgxrG1FxMEOFzZp1rPJ4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@wolflamb632,I loooove lapcheong.,0,0,20,...,i loove lapcheong.,positive,0.973395,positive,0.679386,neutral,0.579830,2,0.8264,majority
5,UgzUAUw_Ih5_mf7848t4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@yangli-q3i,It&#39;s so delicious!,0,0,22,...,it's so delicious!,positive,0.977652,positive,0.864241,positive,0.775318,2,0.8724,full


In [30]:
print(high_conf['label'].value_counts())
print()
print(high_conf['label'].value_counts(normalize=True).mul(100).round(1))

label
2    12127
1     8300
0     4888
Name: count, dtype: int64

label
2    47.9
1    32.8
0    19.3
Name: proportion, dtype: float64


In [ ]:
high_conf.to_csv('../data/labeled_comments.csv', index=False, encoding='utf-8-sig')
print(f"Saved {len(high_conf)} labeled samples")

Saved 25315 labeled samples


### Filter Training Dataset

In [ ]:
food_keywords = [
    'food', 'eat', 'eating', 'restaurant', 'taste', 'ate',
    'cafe', 'dish', 'meal', 'taste', 'cuisine', 
    'sweet', 'sour', 'bitter', 'spicy', 'crispy',
    'delicious', 'burger', 'pizza', 'sauce', 'curry',
    'ramen', 'nasi', 'laksa', 'lapcheong', 'teh', 'bones',
    'meat', 'chicken', 'fish', 'lamb', 'pork', 'beef', 'wings', 'lungs',
    'ikan', 'ayam', 'daging', 'kambing',
    'rice', 'noodle', 'cendol',
    'fruit', 'durian'
]

def food_filter(comment):
    comment = str(comment).lower()
    return any(word in comment for word in food_keywords)

high_conf["food_related"] = high_conf["comment_en"].apply(food_filter)

In [33]:
print(high_conf['food_related'].value_counts())
print()
print(high_conf['food_related'].value_counts(normalize=True).mul(100).round(1))

food_related
True     13092
False    12223
Name: count, dtype: int64

food_related
True     51.7
False    48.3
Name: proportion, dtype: float64


In [34]:
high_conf.to_csv('../data/labeled_comments.csv', index=False, encoding='utf-8-sig')
print(f"Saved {len(high_conf)} labeled samples")

Saved 25315 labeled samples


### Manual Review

In [ ]:
# new_food_keywords = [
#     'restaurant', 'taste',
#     'cafe', 'dish', 'meal', 'try', 'cuisine', 'recipe',
#     'sweet', 'sour', 'bitter', 'spicy', 'crispy', 'tasty', 'yummy',
#     'delicious', 'burger', 'pizza', 'sauce', 'curry', 'satay', 'chapati', 'dessert', 'dim sum', 'tofu', 'stingray', 'lekor', 'kuih',
#     'budu', 'tempoyak', 'fermented', 'roti', 'jelly', 'goreng', 'ice',
#     'smoothies', 'snacks', 'fried', 'bun', 'pandan', 'peanuts', 'apam', 'sambal', 'bread', 'omelette',
#     'ramen', 'nasi', 'laksa', 'lapcheong', 'teh', 'bones',
#     # 'meat', 'chicken', 'fish', 'lamb', 'pork', 'beef', 'wings', 'lungs', 'shirimp', 'squid',
#     'ikan', 'ayam', 'daging', 'kambing', 'udang',
#     'rice', 'noodle', 'mee', 
#     'cendol', 'cempedak', 'mango',
#     'fruit', 'durian'
# ]

In [11]:
df = pd.read_excel("../data/labeled_comments_review.xlsx")
df.head()

,comment_id,video_id,video_title,video_url,channel_title,author,comment,likes,reply_count,comment_length,...,xlmr_label,xlmr_score,multi_label,multi_score,label,confidence,agreement,food_related,food_related_review,label_review
0,UgzaWfPbrkBJ2IAnmnd4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@spicytrinirice,Mark how are you so skinny?,0,0,27,...,negative,0.879051,neutral,0.673817,1,0.6511,majority,False,False,1
1,Ugw_Y1XANSnQcxzrG1x4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@WaqasAli-dq9wg,Kuala Lumpur rocks! I think I&#39;m gonna be r...,0,0,105,...,positive,0.891053,positive,0.585007,2,0.8218,full,False,False,2
2,Ugy8SZOGQ-vroxWT0QV4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@AhhiongChia,Why not batu Kawa,0,0,17,...,neutral,0.484825,neutral,0.512170,1,0.5504,full,False,False,1
3,UgwfqJyg_z4KM8ValER4AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@elenbadiana6354,Where is this?,0,0,14,...,neutral,0.683180,neutral,0.889595,1,0.8185,full,False,False,1
4,UgyXq1IuWqodcnRgyF94AaABAg,rakfl1KCaMk,I Ate the Best Malaysian STREET FOOD for 24 Ho...,https://www.youtube.com/watch?v=rakfl1KCaMk,Mark Wiens,@CarlosAlbertoZelaya-y4u,No thanks but thanks....<br><br>Nasty....,0,0,41,...,negative,0.821997,negative,0.478640,0,0.6906,full,False,False,0


In [12]:
counts = (
    df.groupby(["food_related", "food_related_review"])
      .size()
      .unstack(fill_value=0)
)
print(counts)

food_related_review  False  True 
food_related                     
False                11757    466
True                  4055   9037


In [17]:
clean_df = df.loc[
    df["food_related_review"],
    ["comment_en", "label_review"]
].drop_duplicates(subset=["comment_en"]).copy()

clean_df.head()

,comment_en,label_review
9,i know everyone in malaysia that can cook one ...,2
12,is there a crispy crust?,1
14,those bones i cant deal with,0
20,try satay bro,1
51,my fav thing to have in the morning is chapati 😍,2


In [20]:
print(clean_df.count())
print()
print(clean_df['label_review'].value_counts())
print()
print(clean_df['label_review'].value_counts(normalize=True).mul(100).round(1))

comment_en      9499
label_review    9499
dtype: int64

label_review
2    4507
1    2702
0    2290
Name: count, dtype: int64

label_review
2    47.4
1    28.4
0    24.1
Name: proportion, dtype: float64


In [23]:
clean_df = clean_df.rename(columns={'comment_en': 'comment', 'label_review': 'label'})
clean_df.head()

,comment,label
9,i know everyone in malaysia that can cook one ...,2
12,is there a crispy crust?,1
14,those bones i cant deal with,0
20,try satay bro,1
51,my fav thing to have in the morning is chapati 😍,2


In [24]:
clean_df.to_csv('../data/labeled_comments_clean.csv', index=False, encoding='utf-8-sig')
print(f"Saved {len(clean_df)} labeled samples")

Saved 9499 labeled samples
